# Titanic Machine Learning Capstone Project

**Student name:** Mazen  
**Course:** Machine Learning, Deep Learning & NLP Applications  
**Dataset:** Titanic Dataset

## Project idea

In this project, I am using Titanic passenger information to predict if a passenger survived or not.

I chose this dataset because it is simple to understand, and the columns actually make sense, like age, gender, ticket class, and fare.

## Step 1.1 - Loading and checking the data

First, I load the dataset and take a quick look at it. I want to know how many rows it has, which columns have missing values, and whether the survival classes are balanced or not.

In [ ]:
# I import the libraries I need for data, charts, models, and scores
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.neural_network import MLPClassifier

# Load the Titanic dataset
df = pd.read_csv('data/titanic.csv')

# Show the first 5 passengers
df.head()

In [ ]:
# Check the size of the dataset
print('Shape:', df.shape)

# Check the column types
print(df.dtypes)

In [ ]:
# Check missing values
print(df.isnull().sum())

In [ ]:
# Check class balance: 0 = did not survive, 1 = survived
print(df['Survived'].value_counts())
print(df['Survived'].value_counts(normalize=True))

df['Survived'].value_counts().plot(kind='bar')
plt.title('Class Balance: Survived vs Not Survived')
plt.xlabel('Survived')
plt.ylabel('Number of passengers')
plt.show()

## Step 1.2 - Preparing the data

For this project, I used columns that are easy for me to explain:

- `Pclass`: ticket class
- `Sex`: male or female
- `Age`: passenger age
- `SibSp`: siblings or spouse on the ship
- `Parch`: parents or children on the ship
- `Fare`: ticket price
- `Embarked`: where the passenger boarded

The answer I want to predict is `Survived`.

In [ ]:
# I chose these columns because they are clear and easy to explain
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
label = 'Survived'

# Keep only the columns I need
data = df[features + [label]].copy()

# Fill missing values
# I use the median age because it is a safe middle value
data['Age'] = data['Age'].fillna(data['Age'].median())

# I use the most common boarding place for missing Embarked values
data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])

# Convert text columns into numbers, because models understand numbers better than words
data = pd.get_dummies(data, columns=['Sex', 'Embarked'], drop_first=True)

data.head()

In [ ]:
# Separate X and y
X = data.drop('Survived', axis=1)
y = data['Survived']

# Split into train and test sets
# 80% train and 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training rows:', X_train.shape[0])
print('Testing rows:', X_test.shape[0])

In [ ]:
# Scale the data so models like KNN and neural networks work better
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Step 1.3 - Training the machine learning models

Now I train three different models and compare them. I am doing this because one model might work better than another on the same dataset.

The models are:

1. Logistic Regression
2. Random Forest
3. KNN

In [ ]:
# Create the models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

results = []

for name, model in models.items():
    # Train the model
    model.fit(X_train_scaled, y_train)

    # Predict on the test data
    y_pred = model.predict(X_test_scaled)

    # Save scores
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-score': f1_score(y_test, y_pred)
    })

    # Show confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Survived', 'Survived'])
    disp.plot()
    plt.title(name + ' Confusion Matrix')
    plt.show()

# Show results table
results_df = pd.DataFrame(results)
results_df

In [ ]:
# Find the best machine learning model using F1-score
best_ml_model = results_df.sort_values('F1-score', ascending=False).iloc[0]
print('Best ML model:', best_ml_model['Model'])
print(best_ml_model)

## Step 2 - Simple neural network

For the neural network part, I used a simple model with hidden layers. I kept it simple on purpose because I want to be able to explain it clearly.

A neural network learns patterns from the data using connected layers.

In [ ]:
# Build a simple neural network
# hidden_layer_sizes=(16, 8) means the model has two hidden layers
# I used this because it is easier to run and explain than a very large neural network

nn_model = MLPClassifier(
    hidden_layer_sizes=(16, 8),
    activation='relu',
    max_iter=500,
    random_state=42
)

# Train the neural network
nn_model.fit(X_train_scaled, y_train)

# Make predictions
nn_pred = nn_model.predict(X_test_scaled)

# Calculate the scores
nn_results = {
    'Model': 'Neural Network',
    'Accuracy': accuracy_score(y_test, nn_pred),
    'Precision': precision_score(y_test, nn_pred),
    'Recall': recall_score(y_test, nn_pred),
    'F1-score': f1_score(y_test, nn_pred)
}

nn_results

In [ ]:
# Confusion matrix for the neural network
cm = confusion_matrix(y_test, nn_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Survived', 'Survived'])
disp.plot()
plt.title('Neural Network Confusion Matrix')
plt.show()

In [ ]:
# Plot the loss curve of the neural network
# Loss should usually go down as the model trains
plt.plot(nn_model.loss_curve_)
plt.title('Neural Network Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

In [ ]:
# Add neural network result to the comparison table
final_results_df = pd.concat([results_df, pd.DataFrame([nn_results])], ignore_index=True)
final_results_df = final_results_df.sort_values('F1-score', ascending=False)
final_results_df

In [ ]:
# Save the results table
final_results_df.to_csv('outputs/model_results.csv', index=False)
print('Saved results to outputs/model_results.csv')

## Step 3 - Final explanation

The best model is the one with the highest F1-score.

I used F1-score because it combines precision and recall, so it gives a better overall idea of the model's performance.

What I learned from this project is that machine learning has many steps. The model is important, but cleaning the data and comparing results are also very important.